In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
import torch
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")

In [ ]:
from utils import olc_client
c = olc_client.connect(verbose=True)

In [ ]:
import numpy as np
import pandas as pd
import scipy
import gc

import connectome_interpreter as ci

In [ ]:
from utils.config import (DATA_DIR, DATASET)

cache_dir = Path(PROJECT_ROOT, 'cache', 'quan_propagation')
cache_dir.mkdir(parents=True, exist_ok=True)

# Load meta

In [ ]:
# LOAD Judith meta, this has AN's R7/8, 
meta = pd.read_csv(DATA_DIR / f'{DATASET}_meta.csv')

#  add superclass and class
meta.loc[meta['cell_type'].str.contains('^R7|^R8'), 'superclass'] = 'ol_sensory'
meta.loc[meta['cell_type'].str.contains('^R7|^R8'), 'class'] = 'visual'

print(meta.shape)  

In [ ]:
# make L1 excitatory so it's an ON cell 
meta.loc[meta.cell_type == 'L1', 'sign'] = 1

In [ ]:
# make dictionaries that map indices to meta info
# idx_to_idx = dict(zip(meta.idx, meta.idx))
idx_to_bodyId = dict(zip(meta.idx, meta.bodyId))
# combine bodyId and cell_type_side
# idx_to_bodyId_cellType = dict(zip(meta.idx, meta.bodyId.astype(str) + '_' + meta.cell_type))
# idx_to_type = dict(zip(meta.idx, meta.cell_type))
# idx_to_sign = dict(zip(meta.idx, meta.sign))
# idx_to_side = dict(zip(meta.idx, meta.side))
# # idx_to_side = dict(zip(meta.idx, meta.soma_side))
idx_to_coords = dict(zip(meta.idx, meta.coords))

# type_to_nt = dict(zip(meta.cell_type, meta.nt))
# root_to_type = dict(zip(meta.bodyId, meta.cell_type))
# idx_to_root = dict(zip(meta.idx, meta.bodyId))
# type_to_sign = {atype:idx_to_sign[idx] for idx, atype in idx_to_type.items()}

# bodyId_to_idx = dict(zip(meta.bodyId, meta.idx))

# idx_to_modality = dict(zip(meta.idx, meta.superclass))

# sign_to_color = {1: '#EE672D', -1: '#1F4695', 0: '#979DA5'}

In [ ]:
# remove all vidual inputs
outidx = meta[
    ~meta['cell_type'].isin(['L1', 'L2', 'L3', 'R7', 'R8', 'R7d', 'R8d','HBeyelet', 'R7R8_unclear', 'R7_unclear', 'R8_unclear'])
    ]['idx']

# effwt, right side

## load stepsn

In [ ]:
stepsn = scipy.sparse.load_npz(Path(DATA_DIR, f'{DATASET}_r_lat_flow_sum.npz'))
stepsn.shape

## effwt matrix from all 5 channels

In [ ]:
inidx = meta.idx[meta.instance.isin(['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R'])] 
# # inidx = meta.idx[meta.cell_type_side.isin(['T4a_R', 'T4b_R', 'T4c_R', 'T4d_R', 'T5a_R', 'T5b_R', 'T5c_R', 'T5d_R'])] 
# outidx = meta[meta['instance'].isin(oltypes0['instance']) &
#               ~meta['cell_type'].isin(['L1', 'L2', 'L3', 'R7', 'R8', 'R7d', 'R8d','HBeyelet', 'R7', 'R8'])
#               ]['idx']
# outidx = meta.idx

effwt = ci.result_summary(stepsn, inidx, outidx,
                    inidx_map = idx_to_coords, 
                    outidx_map = idx_to_bodyId,
                    display_threshold = 0,
                    display_output= False
                    )
# # effwt_visr.columns = effwt_visr.columns.astype(float).astype(int)
# effwt = effwt[effwt.index != 'nan']
print(effwt.shape)

# save effwt_visr to pickle
effwt.to_pickle(Path(cache_dir, 'effwt_visr_5.pkl'))

# load effwt_visr from pickle
# effwt_visr = pd.read_pickle(Path(cache_dir, 'effwt_visr_5.pkl'))


## effwt matrix from individual channels

In [ ]:
channel_list = ['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R', 'R7d_R', 'R8d_R', 'HBeyelet_R']
meta[meta.instance.isin(channel_list)]['instance'].value_counts()

In [ ]:
channel_list = ['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R', 'R7d_R', 'R8d_R', 'HBeyelet_R']

for ch in channel_list:
    inidx = meta.idx[meta.instance.isin([ch])] 
    effwt = ci.result_summary(stepsn, inidx, outidx,
                        inidx_map = idx_to_coords, 
                        outidx_map = idx_to_bodyId,
                        display_threshold = 0,
                        display_output= False
                        )
    print(effwt.shape)

    # save effwt_visr to pickle
    effwt.to_pickle(Path(cache_dir, f'effwt_visr_{ch}.pkl'))

In [ ]:
del stepsn
gc.collect()

# effwt, left side

## load stepsn

In [ ]:
stepsn = scipy.sparse.load_npz(Path(DATA_DIR, f'{DATASET}_l_lat_flow_sum.npz'))
stepsn.shape

## effwt matrix from all 5 channels

In [ ]:
inidx = meta.idx[meta.instance.isin(['L1_L', 'L2_L', 'L3_L', 'R7_L', 'R8_L'])] 
effwt = ci.result_summary(stepsn, inidx, outidx,
                    inidx_map = idx_to_coords, 
                    outidx_map = idx_to_bodyId,
                    display_threshold = 0,
                    display_output= False
                    )
print(effwt.shape)

# save effwt_visl to pickle
effwt.to_pickle(Path(cache_dir, 'effwt_visl_5.pkl'))

# load effwt_visl from pickle
# effwt_visl = pd.read_pickle(Path(cache_dir, 'effwt_visl_5.pkl'))

In [ ]:
del stepsn
gc.collect()

# End